In [ ]:
:set -XRankNTypes
:set -XScopedTypeVariables
putStrLn "Setup done."

# 🚀 GPU / Accelerate в Haskell

**Разделы:** категорный взгляд → Accelerate EDSL → Repa → стадийная компиляция → fusion

**Уровень:** специализированный | **Предварительные требования:** [Монады](Monads.ipynb), [Метапрограммирование](MetaProgramming.ipynb)

![Ландшафт GPU](../diagrams/haskell/gpu_landscape.svg)

---

> **📦 Зависимости**
> **Пакеты:** `array`
> **Расширения:** `RankNTypes` — forall внутри аргументов функций ([→](Extensions.ipynb#rankntypes)) · `ScopedTypeVariables` — переменные типа из сигнатуры видны в теле ([→](Extensions.ipynb#scopedtypevariables))


❖Содержание

| # | Тема | Суть |
|---|------|------|
| 1 | 1️⃣ 📊 Данные-параллелизм: категорная основа |  |
| 2 | 2️⃣ Accelerate EDSL: стадийная компиляция |  |
| 3 | 3️⃣ Stencil = комонада |  |
| 4 | 4️⃣ Repa: отложенные массивы + fusion |  |
| 5 | 5️⃣ Категорный взгляд: единая картина |  |

## 1️⃣ 📊 Данные-параллелизм: категорная основа

**Данные-параллелизм** — одна и та же операция применяется к большому количеству данных одновременно.

### Array = Экспоненциальный объект в категории

`Array sh e` ≅ `sh -> e` — массив = функция из индекса в элемент.

| Операция | Категорный смысл | Пример |
|-----------|------------------|---------|
| `map f` | fmap :: Functor | `map (+1) xs` |
| `fold f z` | catamorphism | `fold (+) 0 xs` |
| `scan f z` | prefix catamorphism | `scan (+) 0 xs` |
| `zipWith f` | liftA2 :: Applicative | `zipWith (*) xs ys` |
| `stencil` | comonad extend | `stencil blur3x3 xs` |
| `permute` | изоморфизм форм | `permute f xs` |

### Fusion = закон функтора

```haskell
-- Fusion автоматически сливает цепочки
map f . map g = map (f . g)  -- один проход по GPU!
fold f z . map g = foldMap (f . g)  -- слить deforestation
```

In [ ]:
-- Array как функтор: законы fmap через чистые массивы
import Data.Array (Array, listArray, bounds, elems, indices, (!))
import qualified Data.Array as A

-- 1D массив = экспоненциальный объект: [0..n-1] -> a
type Vec a = Array Int a

mkVec :: [a] -> Vec a
mkVec xs = listArray (0, length xs - 1) xs

-- fmap = параллельный map (функтор!)
vmapF :: (a -> b) -> Vec a -> Vec b
vmapF f arr = A.array (bounds arr) [(i, f (arr ! i)) | i <- indices arr]

-- fold = катаморфизм
vfold :: (b -> a -> b) -> b -> Vec a -> b
vfold f z arr = foldl f z (elems arr)

-- zipWith = liftA2
vzipWith :: (a -> b -> c) -> Vec a -> Vec b -> Vec c
vzipWith f a b
  | bounds a /= bounds b = error "shape mismatch"
  | otherwise = A.array (bounds a) [(i, f (a!i) (b!i)) | i <- indices a]

-- scan (префиксный катаморфизм)
vscanl :: (b -> a -> b) -> b -> Vec a -> Vec b
vscanl f z arr =
  let (lo, hi) = bounds arr
      go i acc
        | i > hi    = []
        | otherwise = let v = f acc (arr ! i)
                      in v : go (i+1) v
  in mkVec (z : go lo z)

-- Fusion закон: map f . map g = map (f . g)
demoFusion :: IO ()
demoFusion = do
  let xs = mkVec [1..10 :: Int]
  
  -- Без fusion: два прохода
  let result1 = vmapF (*2) (vmapF (+1) xs)
  
  -- С fusion: один проход
  let result2 = vmapF ((*2) . (+1)) xs
  
  putStrLn $ "Without fusion: " ++ show (elems result1)
  putStrLn $ "With fusion: " ++ show (elems result2)
  putStrLn $ "Equal (fusion law): " ++ show (elems result1 == elems result2)
  
  -- zipWith = Applicative liftA2
  let ys = mkVec [10,20..100 :: Int]
  let dotProduct = vfold (+) 0 (vzipWith (*) xs ys)
  putStrLn $ "\nDot product: " ++ show dotProduct
  
  -- Prefix scan
  let prefixSums = vscanl (+) 0 xs
  putStrLn $ "Prefix sums: " ++ show (elems prefixSums)

demoFusion

## 2️⃣ Accelerate EDSL: стадийная компиляция

**Accelerate** — глубоко вложенный DSL: Haskell → AST → LLVM → GPU.

```haskell
-- Accelerate: два уровня стадий
type Acc a   -- GPU выражения (массивы)
type Exp e   -- скалярные GPU выражения

-- Компиляция и запуск
run  :: Acc a -> a           -- компилирует + запускает
run1 :: (Acc a -> Acc b) -> a -> b  -- один аргумент

-- Операции
map      :: (Exp a -> Exp b) -> Acc (Array sh a) -> Acc (Array sh b)
fold     :: (Exp a -> Exp a -> Exp a) -> Exp a -> Acc (Array (sh:.Int) a) -> Acc (Array sh a)
zipWith  :: (Exp a -> Exp b -> Exp c) -> Acc (Array sh a) -> Acc (Array sh b) -> Acc (Array sh c)
stencil  :: Stencil sh a stencil => (stencil -> Exp b) -> Boundary a -> Acc (Array sh a) -> Acc (Array sh b)
```

### Стадийная компиляция = метапрограммирование

| Стадия | Что вычисляется |
|-------|------------------|
| 1 (compile-time GHC) | Типизация Acc/Exp, построение AST |
| 2 (Accelerate runtime) | Fusion, оптимизация, кодогенерация LLVM/CUDA |
| 3 (GPU hardware) | Выполнение кернелов |

In [ ]:
-- Стадийные вычисления через GADT
-- Симулируем двухуровневую стадийную AST (как в Accelerate)
{-# LANGUAGE GADTs #-}

-- AST скалярных выражений (Exp)
data Expr a where
  Lit    :: Int -> Expr Int
  Add    :: Expr Int -> Expr Int -> Expr Int
  Mul    :: Expr Int -> Expr Int -> Expr Int
  If     :: Expr Bool -> Expr a -> Expr a -> Expr a
  BoolLit :: Bool -> Expr Bool
  Gt     :: Expr Int -> Expr Int -> Expr Bool

-- Интерпретация (вычислить выражение)
evalExpr :: Expr a -> a
evalExpr (Lit n)      = n
evalExpr (Add a b)    = evalExpr a + evalExpr b
evalExpr (Mul a b)    = evalExpr a * evalExpr b
evalExpr (BoolLit b)  = b
evalExpr (Gt a b)     = evalExpr a > evalExpr b
evalExpr (If c t e)   = if evalExpr c then evalExpr t else evalExpr e

-- Кодогенерация (стадия 2: генерировать код)
codegenExpr :: Expr a -> String
codegenExpr (Lit n)     = show n
codegenExpr (Add a b)   = "(" ++ codegenExpr a ++ " + " ++ codegenExpr b ++ ")"
codegenExpr (Mul a b)   = "(" ++ codegenExpr a ++ " * " ++ codegenExpr b ++ ")"
codegenExpr (BoolLit b) = if b then "true" else "false"
codegenExpr (Gt a b)    = "(" ++ codegenExpr a ++ " > " ++ codegenExpr b ++ ")"
codegenExpr (If c t e)  = "if " ++ codegenExpr c ++ " then " ++ codegenExpr t ++ " else " ++ codegenExpr e

-- Fusion на AST: вольная теорема (deforestation)
fuseExpr :: Expr a -> Expr a
fuseExpr (Add (Lit 0) e) = fuseExpr e       -- 0 + e = e
fuseExpr (Add e (Lit 0)) = fuseExpr e       -- e + 0 = e
fuseExpr (Mul (Lit 1) e) = fuseExpr e       -- 1 * e = e
fuseExpr (Mul e (Lit 1)) = fuseExpr e       -- e * 1 = e
fuseExpr (Add a b)       = Add (fuseExpr a) (fuseExpr b)
fuseExpr (Mul a b)       = Mul (fuseExpr a) (fuseExpr b)
fuseExpr e               = e

demoStagedGPU :: IO ()
demoStagedGPU = do
  -- Стадия 1: построить AST (как Acc/Exp в Accelerate)
  let expr = Mul (Lit 1) (Add (Lit 0) (Add (Lit 3) (Mul (Lit 4) (Lit 5))))
  putStrLn $ "Stage 1 AST: " ++ codegenExpr expr
  
  -- Стадия 2: оптимизация (fusion / constant folding)
  let fused = fuseExpr expr
  putStrLn $ "Stage 2 fused: " ++ codegenExpr fused
  
  -- Стадия 3: вычислить
  putStrLn $ "Stage 3 result: " ++ show (evalExpr fused)
  
  -- GPU перспектива: применить к каждому элементу массива
  let arrExprs = map (\x -> fuseExpr (Mul (Lit 1) (Add (Lit x) (Lit 10)))) [1..5]
  let results = map evalExpr arrExprs
  putStrLn $ "\nGPU-style map over array: " ++ show results

demoStagedGPU

## 3️⃣ Stencil = комонада

**Stencil** — операция над массивом, где каждый элемент вычисляется с учётом соседей.

```
stencil :: (Stencil sh a s) => (s -> Exp b) -> Boundary a -> Acc (Array sh a) -> Acc (Array sh b)
```

**Комонада** `Comonad w` имеет:
- `extract :: w a -> a` — текущее значение
- `extend :: (w a -> b) -> w a -> w b` — применить функцию с контекстом

Stencil = `extend f arr`, где `f :: NeighborView a -> b` (функция от окна).

**Примеры stencil операций:**
- Сглаживание (блюр): `[1/3, 1/3, 1/3]`
- Фильтр Собеля: вычисление градиентов
- Game of Life: подсчёт живых соседей

In [ ]:
-- Stencil как комонадатическое extend
import Data.Array (Array, listArray, bounds, elems, indices, (!))
import qualified Data.Array as A

type Vec a = Array Int a

mkVec :: [a] -> Vec a
mkVec xs = listArray (0, length xs - 1) xs

-- Контекст элемента (для extend)
data Focus a = Focus
  { focusArray :: Vec a
  , focusIndex :: Int
  } deriving Show

-- Извлечь: extract = текущее значение
extract :: Focus a -> a
extract (Focus arr i) = arr ! i

-- extend: применить функцию к каждому окну
extend :: (Focus a -> b) -> Vec a -> Vec b
extend f arr = A.array (bounds arr) [(i, f (Focus arr i)) | i <- indices arr]

-- Получить соседей с граничными условиям
nbr :: Int -> Focus a -> Maybe a
nbr delta (Focus arr i) =
  let j = i + delta
      (lo, hi) = bounds arr
  in if j >= lo && j <= hi then Just (arr ! j) else Nothing

-- Stencil: сглаживание (блюр) = среднее 3 соседей
blur1D :: Focus Double -> Double
blur1D f =
  let v = extract f
      l = maybe v id (nbr (-1) f)
      r = maybe v id (nbr   1  f)
  in (l + v + r) / 3.0

-- Stencil: детекция границ (фильтр Лапласа)
laplacian1D :: Focus Double -> Double
laplacian1D f =
  let v = extract f
      l = maybe v id (nbr (-1) f)
      r = maybe v id (nbr   1  f)
  in l - 2*v + r

-- Game of Life: stencil на 1D клеточном автомате
golStep :: Focus Int -> Int
golStep f =
  let v = extract f
      l = maybe 0 id (nbr (-1) f)
      r = maybe 0 id (nbr   1  f)
      neighbors = l + r
  in case (v, neighbors) of
       (1, 2) -> 1  -- живая, 2 соседа -> выживает
       (1, _) -> 0  -- живая, другое -> умирает
       (0, 1) -> 1  -- мёртвая, 1 сосед -> оживает
       _      -> 0

demoStencil :: IO ()
demoStencil = do
  -- 1D blur
  let signal = mkVec [0, 0, 0, 10, 0, 0, 0 :: Double]
  let blurred = extend blur1D signal
  putStrLn $ "Signal: " ++ show (elems signal)
  putStrLn $ "Blurred: " ++ show (map (\x -> fromIntegral (round (x * 10)) / 10.0) $ elems blurred)
  
  -- Laplacian
  let lapl = extend laplacian1D signal
  putStrLn $ "Laplacian: " ++ show (elems lapl)
  
  -- Game of Life: 1D Rule 110
  let life = mkVec [0, 1, 1, 0, 1, 0, 1, 1, 0 :: Int]
  let step1 = extend golStep life
  let step2 = extend golStep step1
  putStrLn $ "\nGame of Life (1D):"
  putStrLn $ "  Gen 0: " ++ show (elems life)
  putStrLn $ "  Gen 1: " ++ show (elems step1)
  putStrLn $ "  Gen 2: " ++ show (elems step2)

demoStencil

## 4️⃣ Repa: отложенные массивы + fusion

**Repa** (Regular Parallel Arrays) — библиотека для параллельных вычислений через автоматическое слияние.

```haskell
-- Repa типы:
Array r sh e  -- r = репрезентация, sh = форма, e = элемент

-- Репрезентации:
D  -- Delayed: отложенный (функция i -> e)
U  -- Unboxed: материализованный в памяти

-- Fusion: отложенные вычисляются только при computeP
map f (map g arr)  -- ни одного прохода по памяти!
computeP :: Monad m => Array D sh e -> m (Array U sh e)  -- запуск
```

### Repa vs Accelerate

| | Repa | Accelerate |
|---|------|------------|
| Платформа | CPU (многоядерный) | GPU (CUDA/OpenCL) |
| Слияние | Delayed arrays | AST-уровень |
| Типы | полиморфные | GADT + встроенные |
| Stencil | через index | stencil API |

Ключевое отличие: в Repa `D` (отложенный) = deforestation = функтор с ленивым pull.

## 5️⃣ Категорный взгляд: единая картина

| Концепция | Категорный объект | Категорный морфизм |
|-----------|------------------|-----------------|
| Array sh e | Экспоненциальный объект | вычисления |
| map | Функтор | функция-морфизм |
| fold | Катаморфизм | рекурсия по форме |
| stencil | Комонада (extend) | вычисление с окном |
| zipWith | Applicative (liftA2) | бинарное действие |
| Fusion | Вольная теорема | закон функтора |

**Ключевые идеи:**
- `Array sh e ≅ sh -> e`: массив = функция от индекса — экспоненциальный объект
- Fusion = map f . map g = map (f . g): один GPU-проход
- Stencil = comonadic extend: контекстное вычисление
- Accelerate AST = GADT с типобезопасной кодогенерацией

![Ландшафт GPU](../diagrams/haskell/gpu_landscape.svg)

---

**← Предыдущий:** [Distributed Haskell](DistributedHaskell.ipynb)  |  **Следующий →** [Топосы](Toposes.ipynb)
